# Task-3 weighting vs mass-matched placebo — controlled training experiment

**Question.** Does *specifically* increasing CORN task-3 loss pressure improve the model's
Grade-3-vs-Grade-4 conditional posterior `p_cond_3 = σ(z₃) = P(y=4 | y∈{3,4}, x)` beyond the
non-specific effect of perturbing the loss by the same amount?

**Why a placebo.** §14 of the research record found `p_cond_3` improved in 9/9 auxiliary-loss runs,
but all nine were compared against the same three frozen NO_RACAF baselines, and RACAF (a
mechanistically inert change) also beat those baselines on `p_cond_3`. The placebo arm P adds the
same weighted loss mass as the treatment T on an unrelated task, so **T − P** isolates the
task-3-specific effect without relying on the baselines at all.

| arm | loss | runs |
|---|---|---|
| **B** | weighted CORN (frozen NO_RACAF seed 42/123/2026 BEST, read-only) | reused |
| **T** | task-3 term × β₃ = 2.0 | seeds 42, 123, 2026 |
| **P** | task-1 term × β₁ = 1 + M₃/M₁ ≈ 1.3792 (same added mass) | seeds 42, 123, 2026 |

**How to use.** Select a GPU runtime and *Run all*. The notebook freezes `PREREGISTRATION.json`
before any training, then trains/evaluates the first unfinished run and continues through all six.
After a disconnect, *Run all* again — completed runs are skipped and interrupted ones resume.
Nothing is ever edited by hand between runs.

In [ ]:
# ==== [1] ENVIRONMENT / IMPORTS ====
import os
import posixpath
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import setup as colab_setup
setup_info = colab_setup.setup()

import colab_config
import verify_environment
env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR, drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"), require_gpu=True)

import csv
import datetime
import gc
import hashlib
import json
import shutil
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_auc_score

import config
import corn
import improved_training_data as itd
import multiseed_runs as msr
import task_weighted_corn as twc
import weighted_corn
from training import (Trainer, TrainingConfig, CheckpointOptions, TrainingStateCheckpoint,
                      expected_policy_name, model_precision_policies)
from training import checkpointing as ckpt
from training.trainer import precision_is_consistent

print("Ready. multiseed_runs protocol:", msr.PROTOCOL_VERSION)

In [ ]:
# ==== [2] CONFIGURATION AND FROZEN PRE-REGISTRATION ====
# Written BEFORE any training. On every later session the stored file is re-verified and compared;
# any difference (a beta, seed, threshold, endpoint or the loss module's source) stops the notebook.

EXPERIMENT_NAME = "Task3WeightingPlacebo"
SEEDS = (42, 123, 2026)                                   # PRE-REGISTERED
BOOTSTRAP_RESAMPLES = 2000                                # PRE-REGISTERED
BOOTSTRAP_SEED = 20260925                                 # PRE-REGISTERED
SEVERE_NPDR_GRADE, PDR_GRADE = 3, 4
MIN_MEAN_GAIN = 0.03                                      # SPECIFIC / IMPROVED bar
NOT_SUPPORTIVE_MEAN = 0.01
MAX_MEAN_QWK_DROP = 0.02
MAX_MEAN_MAE_RISE = 0.03
MIN_MEAN_GRADE3_RECALL_CHANGE = -0.10
MIN_MEAN_G4_VS_REST_CHANGE = -0.02
PERTURBATION_EFFECT_MEAN = 0.03                           # descriptive P-B classification

ARMS = {
    "T": {"task": twc.TREATMENT_TASK, "beta": twc.TREATMENT_BETA,
          "task_weights": list(twc.TREATMENT_TASK_WEIGHTS), "dirname": "T_task3_beta2.0000"},
    "P": {"task": twc.PLACEBO_TASK, "beta": twc.PLACEBO_BETA,
          "task_weights": list(twc.PLACEBO_TASK_WEIGHTS), "dirname": "P_task1_beta1.3792"},
}
MASSES = twc.task_loss_masses()
assert abs((ARMS["T"]["beta"] - 1) * MASSES[3] - (ARMS["P"]["beta"] - 1) * MASSES[1]) < 1e-9
assert round(ARMS["P"]["beta"], 4) == 1.3792 and ARMS["T"]["task_weights"] == [1.0, 1.0, 1.0, 2.0]

EXPERIMENTS_ROOT = colab_config.DRIVE.experiments_root
SIX_RUN_ROOT = posixpath.join(EXPERIMENTS_ROOT, "ImprovedTraining")
SIX_RUN_ID = "improved_multiseed_2026_09"
SIX_RUN_DIR = msr.experiment_root(SIX_RUN_ROOT, SIX_RUN_ID)
PARENT_DIR = posixpath.join(EXPERIMENTS_ROOT, EXPERIMENT_NAME)

# One timestamped experiment directory, created on the first session and reused afterwards.
os.makedirs(PARENT_DIR, exist_ok=True)
_existing = sorted(d for d in os.listdir(PARENT_DIR)
                   if os.path.exists(posixpath.join(PARENT_DIR, d, "PREREGISTRATION.json")))
if len(_existing) > 1:
    raise RuntimeError(f"More than one pre-registered experiment under {PARENT_DIR}: {_existing}. "
                       "Refusing to guess which one to continue.")
EXPERIMENT_TIMESTAMP = _existing[0] if _existing else datetime.datetime.now().strftime(
    "%Y-%m-%d_%H-%M-%S")
EXPERIMENT_ID = f"task3_placebo_{EXPERIMENT_TIMESTAMP}"
EXPERIMENT_DIR = posixpath.join(PARENT_DIR, EXPERIMENT_TIMESTAMP)

for _protected in (SIX_RUN_DIR, posixpath.join(EXPERIMENTS_ROOT, "Grade4AuxLoss"),
                   posixpath.join(EXPERIMENTS_ROOT, "Grade3vs4_Phase0"),
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC1Control"),
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC3Kappa"),
                   colab_config.DRIVE.experiment_dir("FinalClassification")):
    assert posixpath.commonpath([posixpath.normpath(EXPERIMENT_DIR),
                                 posixpath.normpath(_protected)]) != posixpath.normpath(_protected), (
        f"{EXPERIMENT_DIR} resolves under protected path {_protected} -- refusing.")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# The frozen six-run protocol, read from its own pre-registration.
SIX_RUN_PREREG, _ = msr.load_and_verify_preregistration(
    posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
BATCH_SIZE = SIX_RUN_PREREG["batch_size"]
MAX_EPOCHS = SIX_RUN_PREREG["max_epochs"]
LEARNING_RATE = SIX_RUN_PREREG["learning_rate"]
WEIGHT_DECAY = SIX_RUN_PREREG["weight_decay"]
CLASS_WEIGHTS = list(SIX_RUN_PREREG["class_weights"])
MONITOR, MODE = SIX_RUN_PREREG["primary_metric"], "max"
EARLY_STOPPING_PATIENCE = SIX_RUN_PREREG["early_stopping"]["patience"]
REDUCE_LR_PATIENCE = SIX_RUN_PREREG["reduce_lr_on_plateau"]["patience"]
REDUCE_LR_FACTOR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["factor"]
MIN_LR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["min_lr"]
assert np.allclose(CLASS_WEIGHTS, weighted_corn.PREREGISTERED_CLASS_WEIGHTS), \
    "six-run class weights differ from the ones the task masses were computed with"

with open(posixpath.join(SIX_RUN_DIR, "experiment_manifest.json")) as _fh:
    SIX_RUN_POPULATION = json.load(_fh)
EXPECTED_VAL_N = int(SIX_RUN_POPULATION["n_val_yielded"])

LOSS_MODULE_SOURCES = (("task_weighted_corn.py", msr.WHOLE_MODULE),)
LOSS_MODULE_DIGEST = msr.normalized_source_digests("task_weighted_corn.py", msr.WHOLE_MODULE,
                                                   msr.REPO_ROOT)
RUN_ORDER = [(arm, seed) for seed in SEEDS for arm in ("T", "P")]

PREREGISTRATION = {
    "experiment_name": EXPERIMENT_NAME, "experiment_id": EXPERIMENT_ID,
    "question": ("Does specifically increasing CORN task-3 loss pressure improve the model's "
                 "Grade-3-vs-Grade-4 conditional posterior p_cond_3 = sigmoid(z3) beyond the "
                 "non-specific effect of perturbing the loss?"),
    "hypothesis": ("Multiplying only CORN task-3's weighted loss numerator by beta3 = 2 raises the "
                   "validation AUROC of p_cond_3 for Grade 4 vs Grade 3 relative to a placebo that "
                   "adds the same weighted loss mass to task 1, without degrading the ordinal task."),
    "motivation_confound": ("Research record section 14: the 9/9 p_cond_3 gain of the auxiliary-loss "
                            "runs was measured against three reused NO_RACAF baselines, and RACAF also "
                            "exceeded those baselines on p_cond_3. It is therefore not causal evidence; "
                            "the placebo arm controls for a non-specific perturbation effect."),
    "estimand": ("p_cond_3 = sigmoid(z3) = P(y=4 | y in {3,4}, x) = p_cum_3 / p_cum_2, the model's own "
                 "Grade-3-vs-Grade-4 posterior. p_gt_3 is NOT the primary endpoint."),
    "arms": {
        "B": "frozen improved_multiseed_2026_09 NO_RACAF seed 42/123/2026 BEST (and LAST), read-only, "
             "never retrained; if unavailable the notebook stops",
        "T": {"task": ARMS["T"]["task"], "beta": ARMS["T"]["beta"],
              "task_weights": ARMS["T"]["task_weights"]},
        "P": {"task": ARMS["P"]["task"], "beta": ARMS["P"]["beta"],
              "beta_rounded": round(ARMS["P"]["beta"], 4),
              "beta_formula": "1 + M3 / M1", "task_weights": ARMS["P"]["task_weights"]},
    },
    "loss": ("numerator = sum_i sum_k beta_k * w[y_i] * m_ik * bce_ik; denominator = sum_i sum_k m_ik "
             "(UNWEIGHTED included-pair count, unchanged); all beta_k = 1 reproduces weighted_corn "
             "exactly (tests/test_task_weighted_corn.py)"),
    "task_loss_masses": {"definition": "M_k = sum_c n_c * w_c * 1[c >= k], TRAINING counts only",
                         "train_counts": list(weighted_corn.PREREGISTERED_TRAIN_COUNTS),
                         "class_weights": CLASS_WEIGHTS, "M": list(MASSES),
                         "added_mass_T": (ARMS["T"]["beta"] - 1) * MASSES[3],
                         "added_mass_P": (ARMS["P"]["beta"] - 1) * MASSES[1]},
    "loss_module_source_digest": LOSS_MODULE_DIGEST,
    "seeds": list(SEEDS), "run_order": [f"{a} seed{s}" for a, s in RUN_ORDER],
    "model": "no_racaf_model.build_no_racaf_joint_model_matched_init via msr.build_arm_model "
             "(unchanged architecture, decoder and matched initialisation)",
    "protocol_from_six_run": {
        "protocol_version": SIX_RUN_PREREG["protocol_version"],
        "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "weight_decay_exclude": list(msr.WEIGHT_DECAY_EXCLUDE_NAMES),
        "class_weights": CLASS_WEIGHTS, "monitor": MONITOR, "mode": MODE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE, "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "reduce_lr_factor": REDUCE_LR_FACTOR, "min_lr": MIN_LR, "mixed_precision": True,
        "split_sha256": msr.EXPECTED_SPLIT_SHA256, "no_auxiliary_loss": True, "no_racaf": True,
        "no_cache_regeneration": True},
    "primary_endpoint": ("validation AUROC of p_cond_3 for Grade 4 vs Grade 3 on the grade-3/4 "
                         "validation images, BEST checkpoint (selected by val_QWK, unchanged)"),
    "primary_contrast": "T - P (per seed, same seed)",
    "secondary_contrasts": ["T - B", "P - B"],
    "secondary_endpoints": [
        "AUROC p_gt_3 Grade 4 vs Grade 3", "AUROC p_gt_2 Grade 4 vs Grade 3",
        "AUROC p_gt_3 Grade 4 vs rest", "QWK", "MAE", "accuracy", "balanced accuracy", "macro F1",
        "Grade-3 recall", "Grade-4 recall", "number/proportion of Grade-4 images decoded <= 2",
        "Brier score of p_cond_3 on Grade-3/4 images", "BEST epoch", "all metrics at LAST checkpoint",
        "per-seed paired stratified bootstrap CIs (2000 resamples)"],
    "decision_rule": {
        "SPECIFIC": f"mean(T-P) >= {MIN_MEAN_GAIN} AND T > P in all 3 seeds",
        "IMPROVED": f"mean(T-B) >= {MIN_MEAN_GAIN} AND T > B in all 3 seeds",
        "PRESERVED": (f"T vs B: mean dQWK >= -{MAX_MEAN_QWK_DROP} AND mean dMAE <= +{MAX_MEAN_MAE_RISE} "
                      f"AND mean dGrade3 recall >= {MIN_MEAN_GRADE3_RECALL_CHANGE} AND mean "
                      f"dAUROC(Grade 4 vs rest) >= {MIN_MEAN_G4_VS_REST_CHANGE}"),
        "verdicts": {"SUPPORTIVE": "SPECIFIC and IMPROVED and PRESERVED",
                     "TRADE_OFF": "SPECIFIC and IMPROVED and not PRESERVED",
                     "NONSPECIFIC": "IMPROVED and not SPECIFIC",
                     "NOT_SUPPORTIVE": f"mean(T-B) < {NOT_SUPPORTIVE_MEAN} OR T <= B in >= 2/3 seeds",
                     "INCONCLUSIVE": "all remaining cases"},
        "evaluation_order": ("if IMPROVED: SUPPORTIVE / TRADE_OFF / NONSPECIFIC by SPECIFIC and "
                             "PRESERVED; else NOT_SUPPORTIVE if its condition holds; else INCONCLUSIVE"),
        "perturbation_effect_descriptive": (f"P-B: |mean| >= {PERTURBATION_EFFECT_MEAN} with the same "
                                            "sign in 3/3 seeds -> PERTURBATION_EFFECT_PRESENT, else "
                                            "NO_CLEAR_PERTURBATION_EFFECT (descriptive, not a verdict)"),
        "no_endpoint_switching": ("the verdict is computed only from the primary endpoint and the "
                                  "guardrails above; no secondary endpoint can change it")},
    "bootstrap": {"resamples": BOOTSTRAP_RESAMPLES, "seed": BOOTSTRAP_SEED,
                  "method": "paired, stratified by class, percentile 95% CI, per seed"},
    "stopping_rule": ("identical to the baseline protocol: EarlyStopping on val_QWK (patience "
                      f"{EARLY_STOPPING_PATIENCE}), cap {MAX_EPOCHS} epochs; no extra epochs, no beta "
                      "tuning, no extra seeds, no reruns selected among; a crashed run resumes through "
                      "the existing checkpoint infrastructure; a run that cannot complete makes the "
                      "verdict INCONCLUSIVE. Stop after the six runs."),
}

PREREG_PATH = posixpath.join(EXPERIMENT_DIR, "PREREGISTRATION.json")
_expected = json.loads(json.dumps(PREREGISTRATION))
if os.path.exists(PREREG_PATH):
    _stored, _digest = msr.load_and_verify_preregistration(PREREG_PATH)
    if {k: v for k, v in _stored.items() if k != "frozen_on"} != _expected:
        raise RuntimeError(f"{PREREG_PATH} differs from this notebook's pre-registration (or the loss "
                           "module changed). The pre-registration is frozen; revert the edit.")
    print(f"Pre-registration verified (frozen on {_stored.get('frozen_on')}): {PREREG_PATH}")
else:
    _payload = dict(_expected, frozen_on=datetime.datetime.now().isoformat(timespec="seconds"))
    with open(PREREG_PATH, "w") as _fh:
        _fh.write(json.dumps(_payload, indent=2, sort_keys=True))
    print(f"Pre-registration FROZEN now at {PREREG_PATH}")
print(f"beta3 = {ARMS['T']['beta']}, beta1 = {ARMS['P']['beta']:.10f} "
      f"(added mass {PREREGISTRATION['task_loss_masses']['added_mass_T']:.4f} vs "
      f"{PREREGISTRATION['task_loss_masses']['added_mass_P']:.4f})")

In [ ]:
# ==== [3] PRE-TRAINING LOSS VERIFICATION (runs every session, before any training) ====
# The same properties tests/test_task_weighted_corn.py proves, re-checked here on this runtime's
# TensorFlow so a version difference can never silently change the loss.
_rng = np.random.default_rng(0)
for _trial in range(5):
    _logits = _rng.normal(0, 2, size=(8, corn.NUM_THRESHOLDS)).astype(np.float32)
    _grades = _rng.integers(0, corn.NUM_GRADES, size=8).astype(np.int32)
    _unit = twc.task_weighted_corn_loss_value(_logits, _grades, CLASS_WEIGHTS, twc.UNIT_TASK_WEIGHTS)
    _ref = weighted_corn.weighted_corn_loss_value(_logits, _grades, CLASS_WEIGHTS)
    assert float(_unit) == float(_ref), "unit task weights do not reproduce weighted_corn"
    _var = tf.Variable(_logits)
    for _arm in ("T", "P"):
        with tf.GradientTape(persistent=True) as _tape:
            _base = twc.task_weighted_corn_loss_value(_var, _grades, CLASS_WEIGHTS,
                                                      twc.UNIT_TASK_WEIGHTS)
            _scaled = twc.task_weighted_corn_loss_value(_var, _grades, CLASS_WEIGHTS,
                                                        ARMS[_arm]["task_weights"])
        _gb, _gs = _tape.gradient(_base, _var).numpy(), _tape.gradient(_scaled, _var).numpy()
        _others = [k for k in range(corn.NUM_THRESHOLDS) if k != ARMS[_arm]["task"]]
        assert np.array_equal(_gb[:, _others], _gs[:, _others]), f"{_arm} touches other tasks"
print("Loss verified: unit weights == weighted_corn; T changes only task 3; P changes only task 1; "
      "equal added mass.")

In [ ]:
# ==== [4] MANIFEST AND RUN DISCOVERY ====
# Status is derived from disk every time.
#   COMPLETED_FROZEN_REFERENCE  B: the frozen six-run NO_RACAF artifacts (read-only)
#   COMPLETED                   T/P: stopped, BEST and LAST evaluated
#   NEEDS_EVALUATION            stopped, evaluation not yet (fully) written
#   INTERRUPTED                 checkpoints exist, not stopped -> resumes
#   NOT_STARTED

MANIFEST = []
for _seed in SEEDS:
    MANIFEST.append({"arm": "B", "seed": _seed, "label": f"B seed{_seed}",
                     "dir": posixpath.join(EXPERIMENT_DIR, "B_frozen_reference", f"seed_{_seed}")})
for _arm, _seed in RUN_ORDER:
    MANIFEST.append({"arm": _arm, "seed": _seed, "label": f"{_arm} seed{_seed}",
                     "dir": posixpath.join(EXPERIMENT_DIR, ARMS[_arm]["dirname"], f"seed_{_seed}")})
for _i, _run in enumerate(MANIFEST, start=1):
    _run["index"] = _i


def frozen_baseline_dir(seed):
    return msr.run_dir(SIX_RUN_ROOT, SIX_RUN_ID, "NO_RACAF", seed)


def per_sample_valid(path):
    if not os.path.exists(path):
        return False
    frame = pd.read_csv(path, usecols=["image_id", "true_grade", "predicted_grade", "p_gt_3"])
    return len(frame) == EXPECTED_VAL_N


def evaluated(run_dir):
    ev = posixpath.join(run_dir, "evaluation")
    return all(os.path.exists(posixpath.join(ev, f)) for f in ("metrics_best.json", "metrics_last.json")) \
        and per_sample_valid(posixpath.join(ev, "per_sample_best.csv")) \
        and per_sample_valid(posixpath.join(ev, "per_sample_last.csv"))


def run_status(run):
    if run["arm"] == "B":
        frozen = frozen_baseline_dir(run["seed"])
        ok = (msr.read_stop_decision(frozen) is not None
              and per_sample_valid(posixpath.join(frozen, "evaluation", "per_sample_best.csv")))
        return "COMPLETED_FROZEN_REFERENCE" if ok else "MISSING_FROZEN_REFERENCE"
    if msr.read_stop_decision(run["dir"]) is not None:
        return "COMPLETED" if evaluated(run["dir"]) else "NEEDS_EVALUATION"
    if ckpt.checkpoint_evidence(posixpath.join(run["dir"], "checkpoints")) or msr.read_history(run["dir"]):
        return "INTERRUPTED"
    return "NOT_STARTED"


def write_status_table():
    table = [{"index": r["index"], "label": r["label"], "status": run_status(r),
              "dir": frozen_baseline_dir(r["seed"]) if r["arm"] == "B" else r["dir"]}
             for r in MANIFEST]
    with open(posixpath.join(EXPERIMENT_DIR, "run_status.json"), "w") as fh:
        json.dump({"updated": datetime.datetime.now().isoformat(timespec="seconds"), "runs": table},
                  fh, indent=2)
    return table


for _row in write_status_table():
    print(f"{_row['index']:>2}. {_row['label']:<12} {_row['status']:<28} {_row['dir']}")
_missing = [r["label"] for r in MANIFEST if run_status(r) == "MISSING_FROZEN_REFERENCE"]
if _missing:
    raise RuntimeError(f"Frozen baseline artifacts unavailable for {_missing}. B is never retrained in "
                       "this experiment -- restore the frozen run instead.")

In [ ]:
# ==== [5] DATA AND MODEL SETUP ====
# The existing cache archive is only EXTRACTED. itd.complete_local_cache() is deliberately NOT called:
# it can regenerate a missing entry, and this experiment must never regenerate caches.
TRAIN_ENTRIES, VAL_ENTRIES, SPLIT_SHA256 = msr.verify_split()
assert SPLIT_SHA256 == msr.EXPECTED_SPLIT_SHA256

LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
LOCAL_CACHE_MARKER = "/content/cache/.multiseed_archive_extracted.json"
CACHE_ARCHIVE_DIR = os.path.join(os.path.dirname(config.LOCAL_FEATURE_RESULTS_DIR), "cache_archive")
STAGING_DIR = "/content/checkpoint_staging_task3placebo"
DATA = {"ready": False, "train": None, "val": None}


def ensure_local_cache():
    if DATA["ready"]:
        return
    if not os.path.exists(LOCAL_CACHE_MARKER):
        import joint_cache_archive as jca
        plan = jca.plan_extraction(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                   racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        jca.print_extraction_plan(plan)
        if plan["drive_unreachable"] or not plan["fits"]:
            raise RuntimeError("Refusing to extract the cache archive.")
        extract = jca.extract_archive(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                      min_free_bytes=plan["required_bytes"])
        jca.print_extract(extract)
        if extract["corrupt"] or extract["drive_unreachable"]:
            raise RuntimeError("Archive extraction did not complete. Re-run the notebook.")
        with open(LOCAL_CACHE_MARKER, "w") as fh:
            json.dump({"extracted": datetime.datetime.now().isoformat(timespec="seconds")}, fh)
    DATA["train"] = itd.locally_cached_entries(TRAIN_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    DATA["val"] = itd.locally_cached_entries(VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    if (len(DATA["train"]), len(DATA["val"])) != (int(SIX_RUN_POPULATION["n_train_yielded"]),
                                                   int(SIX_RUN_POPULATION["n_val_yielded"])):
        raise RuntimeError(f"Extracted cache yields {len(DATA['train'])}/{len(DATA['val'])}, pinned "
                           f"{SIX_RUN_POPULATION['n_train_yielded']}/{SIX_RUN_POPULATION['n_val_yielded']}. "
                           "No cache is regenerated here.")
    assert shutil.disk_usage("/content").free >= 3 * 1024 ** 3, "less than 3 GiB free on /content"
    DATA["ready"] = True
    print(f"Population matches the six-run pin: {len(DATA['train'])} train / {len(DATA['val'])} val")


def fresh_session():
    gc.collect()
    tf.keras.backend.clear_session()


def build_model(seed, arm):
    """msr.build_arm_model('NO_RACAF', seed): the frozen baselines' exact construction (seed, mixed
    precision, matched-init NO_RACAF, AdamW, weighted CORN, structural verification). T and P are then
    recompiled with a fresh identical optimizer and the per-task-weighted loss; nothing else changes."""
    model = msr.build_arm_model("NO_RACAF", seed, class_weights=CLASS_WEIGHTS,
                                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, verbose=0)
    if arm in ARMS:
        model.compile(
            optimizer=msr.build_optimizer(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
            loss=twc.make_task_weighted_corn_loss(CLASS_WEIGHTS, ARMS[arm]["task_weights"]),
            metrics=[corn.CORNQuadraticWeightedKappa(), weighted_corn.UnweightedCORNLoss()])
        assert precision_is_consistent(expected_policy_name(True), model_precision_policies(model))
        if expected_policy_name(True) == "mixed_float16":
            assert getattr(model.optimizer, "inner_optimizer", None) is not None, \
                "recompiled optimizer lost its LossScaleOptimizer wrapper"
    return model

In [ ]:
# ==== [6] METRICS AND EVALUATION ====
# The same code scores B, T and P, so every comparison is like-for-like.

def with_p_cond_3(frame):
    frame = frame.copy()
    reconstructed = frame["p_gt_3"] / frame["p_gt_2"].clip(lower=1e-12)
    if "p_cond_3" in frame.columns:
        ok = frame["p_gt_2"] > 1e-6
        assert np.allclose(frame.loc[ok, "p_cond_3"], reconstructed[ok], atol=1e-4), \
            "p_cond_3 inconsistent with p_gt_3 / p_gt_2"
    else:
        frame["p_cond_3"] = reconstructed
    return frame


def per_sample_metrics(frame):
    frame = with_p_cond_3(frame)
    summary = msr.summarize_predictions(frame.to_dict("records"))
    y = frame["true_grade"].to_numpy(dtype=int)
    pred = frame["predicted_grade"].to_numpy(dtype=int)
    m34 = np.isin(y, [SEVERE_NPDR_GRADE, PDR_GRADE])
    y34 = (y[m34] == PDR_GRADE).astype(int)
    cond = frame["p_cond_3"].to_numpy(np.float64)
    n4 = int((y == PDR_GRADE).sum())
    le2 = int(((y == PDR_GRADE) & (pred <= 2)).sum())
    return {
        "auroc_p_cond_3_g4_vs_g3": float(roc_auc_score(y34, cond[m34])),          # PRIMARY
        "auroc_p_gt_3_g4_vs_g3": float(roc_auc_score(y34, frame["p_gt_3"].to_numpy()[m34])),
        "auroc_p_gt_2_g4_vs_g3": float(roc_auc_score(y34, frame["p_gt_2"].to_numpy()[m34])),
        "auroc_p_gt_3_g4_vs_rest": float(roc_auc_score((y == PDR_GRADE).astype(int),
                                                       frame["p_gt_3"].to_numpy())),
        "qwk": summary["qwk"], "mae": summary["mae"], "accuracy": summary["accuracy"],
        "balanced_accuracy": summary["balanced_accuracy"], "macro_f1": summary["macro_f1"],
        "grade3_recall": summary["recall"][SEVERE_NPDR_GRADE],
        "grade4_recall": summary["recall"][PDR_GRADE],
        "grade4_precision": summary["precision"][PDR_GRADE],
        "n_grade4_decoded_le2": le2, "prop_grade4_decoded_le2": le2 / n4 if n4 else None,
        "brier_p_cond_3_g34": float(np.mean((cond[m34] - y34) ** 2)),
        "confusion_matrix": summary["confusion_matrix"],
        "n": int(len(frame)), "n_grade3": int((y == SEVERE_NPDR_GRADE).sum()), "n_grade4": n4,
    }


def write_evaluation(run_dir, which, rows, extra):
    ev = posixpath.join(run_dir, "evaluation")
    os.makedirs(ev, exist_ok=True)
    with open(posixpath.join(ev, f"per_sample_{which}.csv"), "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    metrics = per_sample_metrics(pd.DataFrame(rows))
    metrics.update(extra)
    with open(posixpath.join(ev, f"metrics_{which}.json"), "w") as fh:
        json.dump(metrics, fh, indent=2)
    return metrics


def evaluate_weights(seed, arm, weights_path):
    ensure_local_cache()
    fresh_session()
    model = build_model(seed, arm)
    ckpt.load_model_weights_only(model, weights_path)
    rows = msr.evaluate_arm_from_disk(model, DATA["val"], cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
    del model
    return rows


def evaluate_new_run(run):
    """BEST and LAST weights of a T/P run, evaluated on the pinned validation set and written to that
    run's own evaluation/ directory."""
    slot_dir, pointer = msr.read_best(run["dir"])
    assert slot_dir is not None, f"{run['label']}: no BEST published"
    rows = evaluate_weights(run["seed"], run["arm"], os.path.join(slot_dir, ckpt.MODEL_WEIGHTS_FILENAME))
    m = write_evaluation(run["dir"], "best", rows, {"label": run["label"], "arm": run["arm"],
                                                    "seed": run["seed"], "checkpoint": "BEST",
                                                    "best_epoch_index_0based": pointer["epoch"]})
    last_dir = ckpt.find_resumable_generation(posixpath.join(run["dir"], "checkpoints"), verbose=False)
    assert last_dir is not None, f"{run['label']}: no valid LAST generation"
    rows = evaluate_weights(run["seed"], run["arm"], os.path.join(last_dir, ckpt.MODEL_WEIGHTS_FILENAME))
    write_evaluation(run["dir"], "last", rows, {"label": run["label"], "arm": run["arm"],
                                                "seed": run["seed"], "checkpoint": "LAST",
                                                "generation": os.path.basename(last_dir)})
    print(f"  evaluated {run['label']}: p_cond_3 AUROC (g4 vs g3) = "
          f"{m['auroc_p_cond_3_g4_vs_g3']:.4f}, QWK = {m['qwk']:.4f}")


def baseline_frame(seed, which):
    """B's per-sample predictions. BEST (and LAST when the frozen run saved it) are read from the
    frozen run; a missing frozen LAST is computed read-only from its latest valid generation and
    stored in THIS experiment's B_frozen_reference directory -- never in the frozen run."""
    frozen = posixpath.join(frozen_baseline_dir(seed), "evaluation", f"per_sample_{which}.csv")
    if per_sample_valid(frozen):
        return pd.read_csv(frozen, dtype={"image_id": str}), frozen
    if which == "best":
        raise RuntimeError(f"frozen BEST per-sample file missing: {frozen}")
    local_dir = posixpath.join(EXPERIMENT_DIR, "B_frozen_reference", f"seed_{seed}")
    local = posixpath.join(local_dir, "evaluation", "per_sample_last.csv")
    if not per_sample_valid(local):
        last_dir = ckpt.find_resumable_generation(
            posixpath.join(frozen_baseline_dir(seed), "checkpoints"), verbose=False)
        assert last_dir is not None, f"B seed{seed}: no valid LAST generation in the frozen run"
        rows = evaluate_weights(seed, "B", os.path.join(last_dir, ckpt.MODEL_WEIGHTS_FILENAME))
        write_evaluation(local_dir, "last", rows, {"label": f"B seed{seed}", "arm": "B", "seed": seed,
                                                   "checkpoint": "LAST (frozen run, read-only)",
                                                   "generation": os.path.basename(last_dir)})
    return pd.read_csv(local, dtype={"image_id": str}), local


def run_frame(run, which):
    if run["arm"] == "B":
        return baseline_frame(run["seed"], which)
    path = posixpath.join(run["dir"], "evaluation", f"per_sample_{which}.csv")
    return pd.read_csv(path, dtype={"image_id": str}), path


def history_summary(run_dir):
    history = msr.read_history(run_dir)
    pointer = msr._read_json(posixpath.join(run_dir, "checkpoints", msr.BEST_POINTER_FILENAME)) or {}
    last = history[-1] if history else {}
    return {"best_epoch_index_0based": pointer.get("epoch"), "best_val_qwk": pointer.get("val_QWK"),
            "epochs_trained": last.get("epoch"),
            "stop_reason": (msr.read_stop_decision(run_dir) or {}).get("reason")}

In [ ]:
# ==== [7] AUTOMATIC RUN EXECUTION (T and P only; B is never trained) ====

def acquire_lock_waiting(run_dir):
    while True:
        try:
            msr.acquire_lock(run_dir, owner_id=msr.OWNER_ID)
            return
        except msr.RunLockedError as error:
            print(f"  lock held by another runtime; waiting for it to go stale. ({error})")
            time.sleep(60)


def arm_config_mapping(arm, seed):
    return {
        "protocol_version": SIX_RUN_PREREG["protocol_version"] + "-task-weighted-corn",
        "experiment_id": EXPERIMENT_ID, "arm": f"NO_RACAF_TASK_WEIGHTED_{arm}", "run_seed": seed,
        "split_seed": msr.SPLIT_SEED, "split_sha256": SPLIT_SHA256,
        "model": "joint_stage05_08_no_racaf", "task_weights": list(ARMS[arm]["task_weights"]),
        "weighted_task": ARMS[arm]["task"], "task_beta": ARMS[arm]["beta"],
        "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "weight_decay_exclude": list(msr.WEIGHT_DECAY_EXCLUDE_NAMES),
        "class_weights": list(CLASS_WEIGHTS), "monitor": MONITOR, "mode": MODE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE, "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "reduce_lr_factor": REDUCE_LR_FACTOR, "min_lr": MIN_LR,
    }


def train_arm_run(run):
    """The same sequence the auxiliary-loss experiment used (msr.train_run's own steps, inlined only
    because train_run is gated to the six-run arms): run manifest, sealed-stop check, training-behaviour
    fingerprint (now also covering task_weighted_corn.py), Trainer with resume, two-slot BEST, history,
    stop decision, lock with heartbeat."""
    ensure_local_cache()
    run_dir, seed, arm = run["dir"], run["seed"], run["arm"]
    msr.ensure_run_dir(run_dir)
    mapping = arm_config_mapping(arm, seed)
    config_hash = ckpt.config_hash(mapping)
    manifest_path = posixpath.join(run_dir, msr.RUN_MANIFEST_FILENAME)
    if os.path.exists(manifest_path):
        msr.verify_run_manifest(run_dir, mapping, config_hash)
    else:
        msr.write_run_manifest(manifest_path, mapping, config_hash, repo_dir=colab_config.REPO_DIR)

    sealed = msr._sealed_stop(posixpath.join(run_dir, "checkpoints"), MAX_EPOCHS)
    if sealed is not None:
        msr.write_stop_decision(run_dir, sealed[0], sealed[1])
        return

    fresh_session()
    model = build_model(seed, arm)
    acquire_lock_waiting(run_dir)
    try:
        sources = {}
        for relative_path, symbols in msr.TRAINING_BEHAVIOR_SOURCES + LOSS_MODULE_SOURCES:
            sources.update(msr.normalized_source_digests(relative_path, symbols, msr.REPO_ROOT))
        behaviour_config = {k: v for k, v in mapping.items()
                            if k not in ("experiment_id", "protocol_version")}
        behaviour_config["mixed_precision"] = True
        components = {"fingerprint_version": msr.TRAINING_BEHAVIOR_FINGERPRINT_VERSION,
                      "configuration": behaviour_config,
                      "train_population": msr.population_digest(DATA["train"]),
                      "validation_population": msr.population_digest(DATA["val"]),
                      "sources": sources}
        msr.reconcile_training_behavior(
            run_dir, {"training_behavior_hash": msr._canonical_hash(components),
                      "components": components},
            msr._current_git_commit(colab_config.REPO_DIR), verbose=1)

        trainer = Trainer(TrainingConfig(
            run_dir=run_dir, epochs=MAX_EPOCHS, monitor=MONITOR, mode=MODE, mixed_precision=True,
            resume=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
            reduce_lr_patience=REDUCE_LR_PATIENCE, reduce_lr_factor=REDUCE_LR_FACTOR,
            min_lr=MIN_LR, precision_check="error", repo_dir=colab_config.REPO_DIR,
            checkpoint_options=CheckpointOptions(
                experiment_id=f"{ARMS[arm]['dirname']}/seed_{seed}", config_hash=config_hash,
                dataset_version="aptos2019-joint-cache-v1", staging_dir=STAGING_DIR,
                keep_generations=2, verbose=1)))
        trainer.prepare(model)
        initial_epoch = trainer.resolve_initial_epoch()
        if initial_epoch > 0:
            trainer.restore(model)
            print(f"  resumed {run['label']} at epoch {initial_epoch}")
        if initial_epoch >= MAX_EPOCHS:
            msr.write_stop_decision(run_dir, initial_epoch, "epoch_cap")
            return

        early = next(c for c in trainer.callbacks if isinstance(c, tf.keras.callbacks.EarlyStopping))
        state_callback = next(c for c in trainer.callbacks if isinstance(c, TrainingStateCheckpoint))
        val_ds = itd.make_epoch_dataset(DATA["val"], epoch=0, run_seed=seed, cache_dir=LOCAL_CACHE_DIR,
                                        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR, batch_size=BATCH_SIZE,
                                        augment=False)
        full_history_dir = posixpath.join(run_dir, "history_full")
        os.makedirs(full_history_dir, exist_ok=True)
        for epoch in range(initial_epoch, MAX_EPOCHS):
            msr.heartbeat_lock(run_dir, owner_id=msr.OWNER_ID)
            train_ds = itd.make_epoch_dataset(DATA["train"], epoch=epoch, run_seed=seed,
                                              cache_dir=LOCAL_CACHE_DIR,
                                              racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                              batch_size=BATCH_SIZE, augment=True)
            model.fit(train_ds, validation_data=val_ds, epochs=epoch + 1, initial_epoch=epoch,
                      callbacks=trainer.callbacks, verbose=1)
            generation = state_callback.last_generation_dir
            assert generation is not None, f"epoch {epoch}: no checkpoint generation written"
            state = ckpt.read_state(generation)
            msr.write_epoch_history(run_dir, state)
            with open(posixpath.join(full_history_dir, f"epoch_{state.completed_epoch:04d}.json"),
                      "w") as fh:
                json.dump(dict((state.extra or {}).get("epoch_logs") or {},
                               epoch=state.completed_epoch, learning_rate=state.learning_rate),
                          fh, indent=2)
            if state.best_epoch == epoch:
                msr.publish_best(run_dir, generation, state, repo_dir=colab_config.REPO_DIR, verbose=1)
            if early.stopped_epoch:
                msr.write_stop_decision(run_dir, state.completed_epoch, "early_stopping")
                break
            if state.completed_epoch >= MAX_EPOCHS:
                msr.write_stop_decision(run_dir, state.completed_epoch, "epoch_cap")
                break
    finally:
        msr.release_lock(run_dir, owner_id=msr.OWNER_ID)
        del model


for run in MANIFEST:
    if run["arm"] == "B":
        continue
    status = run_status(run)
    if status == "COMPLETED":
        continue
    print(f"\n=== [{run['index']}/{len(MANIFEST)}] {run['label']} ({status}) ===")
    if status in ("NOT_STARTED", "INTERRUPTED"):
        train_arm_run(run)
    if msr.read_stop_decision(run["dir"]) is not None:
        evaluate_new_run(run)
    write_status_table()
    print(f"  {run['label']}: {run_status(run)}")

print("\nRun status after this session:")
for _row in write_status_table():
    print(f"{_row['index']:>2}. {_row['label']:<12} {_row['status']}")

In [ ]:
# ==== [8] COMPARISON, PAIRED BOOTSTRAP AND THE PRE-REGISTERED VERDICT ====
METRICS = ["auroc_p_cond_3_g4_vs_g3", "auroc_p_gt_3_g4_vs_g3", "auroc_p_gt_2_g4_vs_g3",
           "auroc_p_gt_3_g4_vs_rest", "qwk", "mae", "accuracy", "balanced_accuracy", "macro_f1",
           "grade3_recall", "grade4_recall", "grade4_precision", "n_grade4_decoded_le2",
           "prop_grade4_decoded_le2", "brier_p_cond_3_g34"]
BOOT_SCORES = {"auroc_p_cond_3_g4_vs_g3": ("p_cond_3", "g34"), "auroc_p_gt_3_g4_vs_g3": ("p_gt_3", "g34"),
               "auroc_p_gt_2_g4_vs_g3": ("p_gt_2", "g34"), "auroc_p_gt_3_g4_vs_rest": ("p_gt_3", "rest")}
CONTRASTS = (("T", "P"), ("T", "B"), ("P", "B"))
N_TRAINED = sum(1 for r in MANIFEST if r["arm"] != "B")
DONE = [r for r in MANIFEST if r["arm"] != "B" and run_status(r) == "COMPLETED"]
FINAL = len(DONE) == N_TRAINED

FRAMES, RESULTS = {"best": {}, "last": {}}, {"best": {}, "last": {}}
for run in MANIFEST:
    if run["arm"] != "B" and run_status(run) != "COMPLETED":
        continue
    for which in ("best", "last"):
        frame, source = run_frame(run, which)
        frame = with_p_cond_3(frame).set_index("image_id").sort_index()
        FRAMES[which][(run["arm"], run["seed"])] = frame
        result = per_sample_metrics(frame.reset_index())
        result.update(label=run["label"], arm=run["arm"], seed=run["seed"], checkpoint=which.upper(),
                      source=source)
        result.update(history_summary(frozen_baseline_dir(run["seed"]) if run["arm"] == "B"
                                      else run["dir"]))
        RESULTS[which][(run["arm"], run["seed"])] = result

# identical validation membership and labels for every run
_ref = FRAMES["best"][("B", SEEDS[0])]
for (_arm, _seed), _frame in list(FRAMES["best"].items()) + list(FRAMES["last"].items()):
    assert _frame.index.equals(_ref.index) and (_frame["true_grade"] == _ref["true_grade"]).all(), \
        f"{_arm} seed{_seed}: validation membership differs from B"


def auc_pairs(score, pos, neg):
    diff = score[pos][:, None] - score[neg][None, :]
    return float(((diff > 0) + 0.5 * (diff == 0)).mean())


def paired_bootstrap(seed_value, frame_a, frame_b, score, population):
    y = frame_a["true_grade"].to_numpy(dtype=int)
    if population == "g34":
        idx = np.flatnonzero(np.isin(y, [SEVERE_NPDR_GRADE, PDR_GRADE]))
    else:
        idx = np.arange(len(y))
    pos = idx[y[idx] == PDR_GRADE]
    neg = idx[y[idx] != PDR_GRADE]
    a, b = frame_a[score].to_numpy(np.float64), frame_b[score].to_numpy(np.float64)
    rng = np.random.default_rng([BOOTSTRAP_SEED, seed_value])
    diffs = np.empty(BOOTSTRAP_RESAMPLES)
    for i in range(BOOTSTRAP_RESAMPLES):
        p, n = rng.choice(pos, pos.size), rng.choice(neg, neg.size)
        diffs[i] = auc_pairs(a, p, n) - auc_pairs(b, p, n)
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)), float((diffs > 0).mean())


COMPARISON, BOOTSTRAP = [], []
for which in ("best", "last"):
    for seed in SEEDS:
        for a, b in CONTRASTS:
            ra, rb = RESULTS[which].get((a, seed)), RESULTS[which].get((b, seed))
            if ra is None or rb is None:
                continue
            row = {"checkpoint": which.upper(), "seed": seed, "contrast": f"{a}-{b}"}
            for m in METRICS:
                va, vb = ra[m], rb[m]
                row[f"{a}_{m}"], row[f"{b}_{m}"] = va, vb
                row[f"delta_{m}"] = None if va is None or vb is None else va - vb
            COMPARISON.append(row)
            for m, (score, population) in BOOT_SCORES.items():
                lo, hi, frac = paired_bootstrap(seed, FRAMES[which][(a, seed)], FRAMES[which][(b, seed)],
                                                score, population)
                BOOTSTRAP.append({"checkpoint": which.upper(), "seed": seed, "contrast": f"{a}-{b}",
                                  "metric": m, "delta": row[f"delta_{m}"], "ci_low": lo, "ci_high": hi,
                                  "fraction_resamples_positive": frac})


def deltas(contrast, metric, which="best"):
    return [r[f"delta_{metric}"] for r in COMPARISON
            if r["checkpoint"] == which.upper() and r["contrast"] == contrast]


PRIMARY = "auroc_p_cond_3_g4_vs_g3"
VERDICT = {"final": FINAL, "completed_trained_runs": len(DONE), "expected_trained_runs": N_TRAINED}
if FINAL:
    d_tp, d_tb, d_pb = deltas("T-P", PRIMARY), deltas("T-B", PRIMARY), deltas("P-B", PRIMARY)
    specific = np.mean(d_tp) >= MIN_MEAN_GAIN and all(d > 0 for d in d_tp)
    improved = np.mean(d_tb) >= MIN_MEAN_GAIN and all(d > 0 for d in d_tb)
    guard = {"mean_delta_qwk": float(np.mean(deltas("T-B", "qwk"))),
             "mean_delta_mae": float(np.mean(deltas("T-B", "mae"))),
             "mean_delta_grade3_recall": float(np.mean(deltas("T-B", "grade3_recall"))),
             "mean_delta_auroc_g4_vs_rest": float(np.mean(deltas("T-B", "auroc_p_gt_3_g4_vs_rest")))}
    preserved = (guard["mean_delta_qwk"] >= -MAX_MEAN_QWK_DROP
                 and guard["mean_delta_mae"] <= MAX_MEAN_MAE_RISE
                 and guard["mean_delta_grade3_recall"] >= MIN_MEAN_GRADE3_RECALL_CHANGE
                 and guard["mean_delta_auroc_g4_vs_rest"] >= MIN_MEAN_G4_VS_REST_CHANGE)
    if improved:
        verdict = ("SUPPORTIVE" if specific and preserved else "TRADE_OFF" if specific
                   else "NONSPECIFIC")
    elif np.mean(d_tb) < NOT_SUPPORTIVE_MEAN or sum(d <= 0 for d in d_tb) >= 2:
        verdict = "NOT_SUPPORTIVE"
    else:
        verdict = "INCONCLUSIVE"
    same_sign = all(d > 0 for d in d_pb) or all(d < 0 for d in d_pb)
    perturbation = ("PERTURBATION_EFFECT_PRESENT"
                    if abs(np.mean(d_pb)) >= PERTURBATION_EFFECT_MEAN and same_sign
                    else "NO_CLEAR_PERTURBATION_EFFECT")
    VERDICT.update(verdict=verdict, specific=bool(specific), improved=bool(improved),
                   preserved=bool(preserved), guardrails=guard,
                   T_minus_P=d_tp, T_minus_B=d_tb, P_minus_B=d_pb,
                   mean_T_minus_P=float(np.mean(d_tp)), mean_T_minus_B=float(np.mean(d_tb)),
                   mean_P_minus_B=float(np.mean(d_pb)), perturbation_effect=perturbation)
else:
    VERDICT["verdict"] = f"PARTIAL ({len(DONE)}/{N_TRAINED} trained runs complete) -- no decision yet"

print(f"Trained runs complete: {len(DONE)}/{N_TRAINED}")
print("VERDICT:", VERDICT["verdict"])
if FINAL:
    print(f"  T-P {VERDICT['T_minus_P']} (mean {VERDICT['mean_T_minus_P']:+.4f}) -> SPECIFIC={VERDICT['specific']}")
    print(f"  T-B {VERDICT['T_minus_B']} (mean {VERDICT['mean_T_minus_B']:+.4f}) -> IMPROVED={VERDICT['improved']}")
    print(f"  guardrails {VERDICT['guardrails']} -> PRESERVED={VERDICT['preserved']}")
    print(f"  P-B {VERDICT['P_minus_B']} (mean {VERDICT['mean_P_minus_B']:+.4f}) -> {VERDICT['perturbation_effect']}")

In [ ]:
# ==== [9] OUTPUTS AND FINAL REPORT ====
def fmt(v, d=4):
    return "n/a" if v is None else f"{v:.{d}f}"


def sfmt(v, d=4):
    return "n/a" if v is None else f"{v:+.{d}f}"


def jsonable(o):
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return str(o)


flat = [dict(r) for which in ("best", "last") for r in RESULTS[which].values()]
pd.DataFrame(flat).drop(columns=["confusion_matrix"], errors="ignore").to_csv(
    posixpath.join(EXPERIMENT_DIR, "per_run_results.csv"), index=False)
pd.DataFrame(COMPARISON).to_csv(posixpath.join(EXPERIMENT_DIR, "comparison.csv"), index=False)
pd.DataFrame(BOOTSTRAP).to_csv(posixpath.join(EXPERIMENT_DIR, "bootstrap_results.csv"), index=False)
with open(posixpath.join(EXPERIMENT_DIR, "results.json"), "w") as fh:
    json.dump({"generated": datetime.datetime.now().isoformat(timespec="seconds"),
               "preregistration": PREREGISTRATION, "verdict": VERDICT,
               "per_run": {f"{k[0]} seed{k[1]} {w}": v for w in ("best", "last")
                           for k, v in RESULTS[w].items()},
               "comparison": COMPARISON, "bootstrap": BOOTSTRAP, "run_status": write_status_table(),
               "environment": env_report}, fh, indent=2, default=jsonable)


def boot(seed, contrast, metric, which="BEST"):
    for b in BOOTSTRAP:
        if (b["seed"], b["contrast"], b["metric"], b["checkpoint"]) == (seed, contrast, metric, which):
            return f"{sfmt(b['delta'])} ({sfmt(b['ci_low'], 3)} to {sfmt(b['ci_high'], 3)})"
    return "n/a"


R = RESULTS["best"]
L = [f"# Task-3 weighting vs mass-matched placebo — {'FINAL' if FINAL else 'PARTIAL'} report", "",
     f"Generated {datetime.datetime.now().isoformat(timespec='seconds')} · experiment `{EXPERIMENT_DIR}` · "
     f"trained runs complete **{len(DONE)}/{N_TRAINED}**.", "",
     "## Pre-registration (frozen before training)", "",
     f"- Question: {PREREGISTRATION['question']}",
     f"- Estimand: {PREREGISTRATION['estimand']}",
     f"- T: task 3 × β₃ = {ARMS['T']['beta']}; P: task 1 × β₁ = {ARMS['P']['beta']:.6f} (= 1 + M₃/M₁); "
     f"both add weighted mass {PREREGISTRATION['task_loss_masses']['added_mass_T']:.3f}. "
     "Denominator = unweighted pair count (unchanged). B = frozen NO_RACAF, read-only.",
     f"- Seeds {', '.join(map(str, SEEDS))}; protocol `{SIX_RUN_PREREG['protocol_version']}` otherwise unchanged.",
     f"- Primary endpoint: {PREREGISTRATION['primary_endpoint']}. Primary contrast: T − P.",
     f"- Rules: SPECIFIC = {PREREGISTRATION['decision_rule']['SPECIFIC']}; IMPROVED = "
     f"{PREREGISTRATION['decision_rule']['IMPROVED']}; PRESERVED = {PREREGISTRATION['decision_rule']['PRESERVED']}.",
     "", "## Run status", "", "| # | run | status | source |", "|---|---|---|---|"]
for row in write_status_table():
    L.append(f"| {row['index']} | {row['label']} | {row['status']} | `{row['dir']}` |")

L += ["", "## 1. Pre-registered primary result", ""]
if FINAL:
    L += [f"**Verdict: {VERDICT['verdict']}** (SPECIFIC = {VERDICT['specific']}, IMPROVED = "
          f"{VERDICT['improved']}, PRESERVED = {VERDICT['preserved']}).", "",
          "| seed | B | T | P | T − P | T − B | P − B |", "|---|---|---|---|---|---|---|"]
    for seed in SEEDS:
        L.append(f"| {seed} | {fmt(R[('B', seed)][PRIMARY])} | {fmt(R[('T', seed)][PRIMARY])} | "
                 f"{fmt(R[('P', seed)][PRIMARY])} | {sfmt(R[('T', seed)][PRIMARY] - R[('P', seed)][PRIMARY])} | "
                 f"{sfmt(R[('T', seed)][PRIMARY] - R[('B', seed)][PRIMARY])} | "
                 f"{sfmt(R[('P', seed)][PRIMARY] - R[('B', seed)][PRIMARY])} |")
    L.append(f"| **mean** | | | | {sfmt(VERDICT['mean_T_minus_P'])} | {sfmt(VERDICT['mean_T_minus_B'])} | "
             f"{sfmt(VERDICT['mean_P_minus_B'])} |")
else:
    L.append("Withheld until all six trained runs are complete.")

L += ["", "## 2. T vs P — specificity (primary contrast)", "",
      "Per-seed paired stratified bootstrap (2,000 resamples) of Δ AUROC `p_cond_3`, Grade 4 vs 3:", ""]
for seed in SEEDS:
    L.append(f"- seed {seed}: {boot(seed, 'T-P', PRIMARY)}")
if FINAL:
    L.append(f"\nSPECIFIC = **{VERDICT['specific']}** (mean {sfmt(VERDICT['mean_T_minus_P'])}, T > P in "
             f"{sum(d > 0 for d in VERDICT['T_minus_P'])}/3 seeds; bar mean ≥ +{MIN_MEAN_GAIN} and 3/3).")

L += ["", "## 3. T vs B — improvement", ""]
for seed in SEEDS:
    L.append(f"- seed {seed}: {boot(seed, 'T-B', PRIMARY)}")
if FINAL:
    L.append(f"\nIMPROVED = **{VERDICT['improved']}** (mean {sfmt(VERDICT['mean_T_minus_B'])}, T > B in "
             f"{sum(d > 0 for d in VERDICT['T_minus_B'])}/3 seeds).")

L += ["", "## 4. Preservation guardrails (T vs B, BEST)", ""]
if FINAL:
    g = VERDICT["guardrails"]
    L += ["| guardrail | mean Δ | bar | pass |", "|---|---|---|---|",
          f"| QWK | {sfmt(g['mean_delta_qwk'])} | ≥ −{MAX_MEAN_QWK_DROP} | {g['mean_delta_qwk'] >= -MAX_MEAN_QWK_DROP} |",
          f"| MAE | {sfmt(g['mean_delta_mae'])} | ≤ +{MAX_MEAN_MAE_RISE} | {g['mean_delta_mae'] <= MAX_MEAN_MAE_RISE} |",
          f"| Grade-3 recall | {sfmt(g['mean_delta_grade3_recall'])} | ≥ {MIN_MEAN_GRADE3_RECALL_CHANGE} | "
          f"{g['mean_delta_grade3_recall'] >= MIN_MEAN_GRADE3_RECALL_CHANGE} |",
          f"| AUROC Grade 4 vs rest | {sfmt(g['mean_delta_auroc_g4_vs_rest'])} | ≥ {MIN_MEAN_G4_VS_REST_CHANGE} | "
          f"{g['mean_delta_auroc_g4_vs_rest'] >= MIN_MEAN_G4_VS_REST_CHANGE} |",
          "", f"PRESERVED = **{VERDICT['preserved']}**."]

L += ["", "## 5. Secondary findings (descriptive; cannot change the verdict)", "",
      "BEST checkpoint, per run:", "",
      "| run | p_cond_3 3v4 | p_gt_3 3v4 | p_gt_2 3v4 | g4 vs rest | QWK | MAE | acc | bal acc | macro F1 | "
      "g3 recall | g4 recall | g4 → ≤2 | Brier p_cond_3 | best ep. | epochs |",
      "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for (arm, seed), r in sorted(R.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    L.append(f"| {arm} seed{seed} | {fmt(r[PRIMARY])} | {fmt(r['auroc_p_gt_3_g4_vs_g3'])} | "
             f"{fmt(r['auroc_p_gt_2_g4_vs_g3'])} | {fmt(r['auroc_p_gt_3_g4_vs_rest'])} | {fmt(r['qwk'])} | "
             f"{fmt(r['mae'])} | {fmt(r['accuracy'])} | {fmt(r['balanced_accuracy'])} | {fmt(r['macro_f1'])} | "
             f"{fmt(r['grade3_recall'])} | {fmt(r['grade4_recall'])} | {r['n_grade4_decoded_le2']}/{r['n_grade4']} | "
             f"{fmt(r['brier_p_cond_3_g34'])} | {r.get('best_epoch_index_0based')} | {r.get('epochs_trained')} |")
L += ["", "LAST checkpoint, primary endpoint:", "", "| seed | B | T | P |", "|---|---|---|---|"]
for seed in SEEDS:
    cells = [fmt(RESULTS["last"].get((a, seed), {}).get(PRIMARY)) for a in ("B", "T", "P")]
    L.append(f"| {seed} | " + " | ".join(cells) + " |")
L += ["", "Paired bootstrap CIs for every contrast, AUROC endpoint and checkpoint are in "
      "`bootstrap_results.csv`; all metric deltas are in `comparison.csv`."]

L += ["", "## The §14 confound — does the placebo resolve it?", ""]
if FINAL:
    L += [f"§14's 9/9 `p_cond_3` gain was measured against three reused NO_RACAF baselines that RACAF "
          f"also beat. Here the placebo P (same added loss mass, unrelated task) gives P − B = "
          f"{', '.join(sfmt(d) for d in VERDICT['P_minus_B'])} (mean {sfmt(VERDICT['mean_P_minus_B'])}): "
          f"**{VERDICT['perturbation_effect']}** (descriptive; |mean| ≥ {PERTURBATION_EFFECT_MEAN} with a "
          "consistent sign).", "",
          "The primary contrast T − P does not use the baselines at all, so the specificity verdict is "
          "immune to the confound. P − B measures how much of any change relative to the frozen "
          "baselines is a generic loss-perturbation effect; it is informative about §14 but does not "
          "re-test the auxiliary loss itself (a different intervention)."]
else:
    L.append("Withheld until all six trained runs are complete.")

L += ["", "## 6. Limitations", "",
      "- 39 Grade-3 and 58 Grade-4 validation images: a single AUROC has SE ≈ 0.058, so the rule "
      "requires a mean gain of 0.03 with 3/3 sign consistency. Three seeds cannot confirm effects "
      "much smaller than that.",
      "- The same 97 validation images have been used in every Grade-3/4 analysis of this project "
      "(§8–§14), including the observation that motivated this experiment; there is no external test set.",
      "- Bootstrap CIs capture validation-sample uncertainty only, not training-seed variability.",
      "- BEST is selected by val_QWK for every arm (unchanged protocol), which barely weighs 3↔4.",
      "- GPU training is not bit-deterministic: matched seeds match initialisation and data order, "
      "not the full trajectory.",
      "- Mass matching equalises expected weighted loss weight, not gradient magnitude; the tasks' "
      "per-element losses differ in size.",
      "- One β per arm: a null result at β₃ = 2 does not exclude other values, and no other value is "
      "tested here.", "",
      f"## Frozen verdict: **{VERDICT['verdict']}**", ""]

REPORT_PATH = posixpath.join(EXPERIMENT_DIR, "REPORT.md")
with open(REPORT_PATH, "w", encoding="utf-8") as fh:
    fh.write("\n".join(L) + "\n")
print("Wrote", REPORT_PATH)
for _name in ("PREREGISTRATION.json", "comparison.csv", "bootstrap_results.csv", "per_run_results.csv",
              "results.json", "run_status.json"):
    print("  ", posixpath.join(EXPERIMENT_DIR, _name))